In [1]:
%pip install -q \
  "transformers==4.46.3" \
  "huggingface-hub<1.0" \
  "tf-keras==2.19.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 164.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 132.2 MB/s eta 0:00:00


In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
import transformers
import huggingface_hub
import pandas as pd
import numpy as np

from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("Transformers version:", transformers.__version__)
print("HF Hub version      :", huggingface_hub.__version__)
print("Pandas version      :", pd.__version__)
print("NumPy version       :", np.__version__)
print("GPU devices detected:", tf.config.list_physical_devices("GPU"))

Python version      : 3.12.13
TensorFlow version  : 2.19.0
Transformers version: 4.46.3
HF Hub version      : 0.36.2
Pandas version      : 2.2.2
NumPy version       : 2.0.2
GPU devices detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)

print(ds_info)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.DS4PLG_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.DS4PLG_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.DS4PLG_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.
tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_dir='/root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0',
    file_format=tfrecord,
    download_size=80.23 MiB,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
   

In [4]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() == 1 else "Negative")
    print(text.numpy().decode("utf-8")[:250], "...\n")

Label: Negative
This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline ...

Label: Negative
I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film wa ...



In [5]:
MAX_LENGTH = 256
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased",
    do_lower_case=True
)

print("Tokenizer loaded:", tokenizer.name_or_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Tokenizer loaded: bert-base-uncased


In [6]:
def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    encoded = tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

    return (
        encoded["input_ids"],
        encoded["attention_mask"],
        encoded["token_type_ids"],
    )

In [7]:
def tf_encode(text, label):
    input_ids, attention_mask, token_type_ids = tf.py_function(
        func=encode_review,
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )

    input_ids.set_shape([MAX_LENGTH])
    attention_mask.set_shape([MAX_LENGTH])
    token_type_ids.set_shape([MAX_LENGTH])
    label.set_shape([])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
    }, label

In [8]:
small_train_raw = ds_train.take(4000)
small_test_raw = ds_test.take(1000)

In [9]:
def prepare_dataset(dataset, shuffle=True):
    dataset = dataset.map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        dataset = dataset.shuffle(2000)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

train_ds = prepare_dataset(small_train_raw, shuffle=True)
test_ds = prepare_dataset(small_test_raw, shuffle=False)

In [10]:
for batch_inputs, batch_labels in train_ds.take(1):
    print(batch_inputs.keys())
    print(batch_inputs["input_ids"].shape)
    print(batch_inputs["attention_mask"].shape)
    print(batch_inputs["token_type_ids"].shape)
    print(batch_labels.shape)

dict_keys(['input_ids', 'attention_mask', 'token_type_ids'])
(16, 256)
(16, 256)
(16, 256)
(16,)


In [11]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

tf_model.h5:   0%|          | 0.00/536M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=2e-5,
    epsilon=1e-8
)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

metrics = [
    tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")
]

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=metrics
)

model.summary()

Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_37 (Dropout)        multiple                  0 (unused)
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
Total params: 109483778 (417.65 MB)
Trainable params: 109483778 (417.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [14]:
EPOCHS = 2

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

Epoch 1/2
250/250 [==============================] - 163s 405ms/step - loss: 0.3887 - accuracy: 0.8255 - val_loss: 0.2935 - val_accuracy: 0.8780
Epoch 2/2
250/250 [==============================] - 102s 376ms/step - loss: 0.1799 - accuracy: 0.9335 - val_loss: 0.3194 - val_accuracy: 0.8960


In [15]:
eval_metrics = model.evaluate(test_ds, return_dict=True)

eval_metrics

63/63 [==============================] - 7s 103ms/step - loss: 0.3194 - accuracy: 0.8960


{'loss': 0.31942203640937805, 'accuracy': 0.8960000276565552}

## Evaluation on Held-Out Test Data

After training, I evaluated the fine-tuned TensorFlow/Keras BERT model on the held-out test pipeline using `model.evaluate()`.

The model achieved:

- **Test loss:** 0.3194
- **Test accuracy:** 0.8960

This means the model correctly classified approximately **89.6%** of the evaluation reviews. Since I trained on a reduced subset of 4,000 training examples and evaluated on 1,000 test examples, this is a strong result for a classroom prototype. The result is also very close to the project’s approximate 0.90 benchmark.

However, the validation loss increased slightly from epoch 1 to epoch 2 while training accuracy continued to rise. This suggests mild overfitting, so in a real workflow I would monitor validation loss carefully and consider early stopping.

In [16]:
def predict_sentiment(text: str):
    encoded = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf"
    )

    inputs = {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "token_type_ids": encoded["token_type_ids"],
    }

    outputs = model(inputs, training=False)
    logits = outputs.logits

    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]

    predicted_class = int(probs.argmax())
    confidence = float(probs[predicted_class])

    label = "Positive" if predicted_class == 1 else "Negative"

    return label, confidence

In [17]:
sentences = [
    "The onboarding emails were confusing, but the agent fixed everything politely.",
    "I waited for three days, nobody helped me, and I am extremely disappointed.",
    "The support team solved my issue quickly and were very kind throughout the process."
]

for sentence in sentences:
    label, confidence = predict_sentiment(sentence)
    print(sentence)
    print(f"Prediction: {label} (confidence={confidence:.3f})")
    print()

The onboarding emails were confusing, but the agent fixed everything politely.
Prediction: Positive (confidence=0.729)

I waited for three days, nobody helped me, and I am extremely disappointed.
Prediction: Negative (confidence=0.980)

The support team solved my issue quickly and were very kind throughout the process.
Prediction: Positive (confidence=0.973)



## Inference Results

After evaluating the model, I tested it on three custom customer-support-style sentences.

1. **Mixed sentence:**  
   "The onboarding emails were confusing, but the agent fixed everything politely."  
   Prediction: **Positive**, confidence = **0.729**

   The model predicted positive sentiment, but with lower confidence than the clearly positive example. This makes sense because the sentence contains both negative information ("confusing onboarding emails") and positive information ("the agent fixed everything politely"). In a real support setting, this type of mixed feedback should probably still be reviewed because the customer had a real pain point even though the final interaction was successful.

2. **Negative sentence:**  
   "I waited for three days, nobody helped me, and I am extremely disappointed."  
   Prediction: **Negative**, confidence = **0.980**

   The model confidently predicted negative sentiment. This matches the sentence, which includes clear signs of frustration, delay, lack of help, and disappointment. In a customer-support scenario, this kind of message would be a strong candidate for escalation.

3. **Positive sentence:**  
   "The support team solved my issue quickly and were very kind throughout the process."  
   Prediction: **Positive**, confidence = **0.973**

   The model confidently predicted positive sentiment. This matches the language of the sentence, which includes a successful resolution and positive service experience.

## Reflection Questions

### 1. What lever most improved results?

The most important lever was fine-tuning a pre-trained BERT model instead of training a model from scratch. BERT already has general language understanding from pretraining, so fine-tuning allowed the model to adapt that knowledge to the sentiment classification task with relatively little data.

In this run, I trained on a subset of 4,000 IMDB reviews and evaluated on 1,000 test reviews. Even with this reduced dataset, the model reached **0.8960 accuracy**, which is close to the project’s ~0.90 benchmark. This suggests that transfer learning was the biggest factor behind the strong result.

Other useful levers would include increasing the amount of training data, tuning the learning rate, adjusting the maximum sequence length, and using early stopping to reduce overfitting.

### 2. Where would you add guardrails before deploying this sentiment signal live?

I would add guardrails at several points before using this model in a real customer-support workflow.

First, I would fine-tune and evaluate the model on real customer-support data, not only movie reviews. IMDB reviews and support tickets both contain sentiment, but they are different domains.

Second, I would use confidence thresholds. For example, highly confident negative messages could be escalated automatically, while lower-confidence or mixed messages could be sent to human review.

Third, I would monitor false negatives carefully. In customer support, missing an angry or frustrated customer may be more costly than incorrectly flagging a neutral message.

I would also add human review for sensitive cases, such as billing disputes, legal complaints, threats, harassment, discrimination, medical information, or anything involving account security. Finally, I would monitor data drift over time because customer language and product issues can change.

### 3. Which stakeholders benefit the most?

The support lead benefits because the model can help prioritize frustrated customers and route urgent conversations more quickly.

The product manager benefits because aggregated sentiment can reveal recurring product pain points, confusing features, or areas where users frequently get stuck.

The compliance officer may also benefit if the system helps flag sensitive or high-risk conversations for review. However, compliance would also need the system to be auditable, monitored, and used carefully rather than as a fully automatic decision-maker.